# Day 30: Capstone — From Optics to Deep Learning 🎓

> **Goal:** Apply everything you've learned to a task connected to your optics & computer vision background. Build a complete end-to-end project and reflect on the journey.

---

## 🎯 Capstone Project: Image Quality Assessment

As an optical engineer, you understand image quality intuitively. Let's build a deep learning system that **automatically classifies image quality** — a task directly relevant to camera/lens evaluation.

We'll use CIFAR-10 images with **synthetic degradations** (blur, noise, compression artifacts) to simulate optical imperfections, then train a model to identify the type of degradation.

### Why This Connects to Optics
| Degradation | Optical Analogue |
|---|---|
| Gaussian blur | Defocus / diffraction-limited PSF |
| Motion blur | Camera shake / motion during exposure |
| Gaussian noise | Sensor noise (read noise, shot noise) |
| JPEG artifacts | Lossy compression in image pipeline |
| Clean | Ideal optical system |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import math
import time
from collections import defaultdict
from PIL import Image, ImageFilter
import io

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

---
## 1. Create the Degradation Dataset

We'll build a custom `Dataset` (the pattern from Day 10) that takes clean CIFAR-10 images and applies synthetic degradations. Each clean image produces 5 versions — one per degradation type — so our dataset is 5× larger than the original.

The `DegradedImageDataset` class below demonstrates:
- **Custom Dataset** with `__len__` and `__getitem__` (Day 10)
- **PIL image processing** for blur, noise, and compression
- **Index arithmetic** to map a flat index to (base_image, degradation_type)

In [ ]:
class DegradedImageDataset(Dataset):
    """
    Takes clean images and applies synthetic degradations.
    Classes: 0=Clean, 1=Gaussian Blur, 2=Motion Blur, 3=Noise, 4=JPEG Artifacts
    """

    DEGRADATION_NAMES = ["Clean", "Gaussian Blur", "Motion Blur", "Noise", "JPEG Artifacts"]

    def __init__(self, base_dataset, transform=None):
        self.base_dataset = base_dataset
        self.transform = transform
        self.num_degradations = 5

    def __len__(self):
        return len(self.base_dataset) * self.num_degradations

    def __getitem__(self, idx):
        base_idx = idx // self.num_degradations
        deg_type = idx % self.num_degradations

        img, _ = self.base_dataset[base_idx]  # PIL Image (from raw CIFAR)

        # Apply degradation
        if deg_type == 0:    # Clean
            degraded = img
        elif deg_type == 1:  # Gaussian blur
            degraded = img.filter(ImageFilter.GaussianBlur(radius=2))
        elif deg_type == 2:  # Motion blur (simulated with directional box filter)
            kernel_size = 5
            kernel = np.zeros((kernel_size, kernel_size))
            kernel[kernel_size // 2, :] = 1.0 / kernel_size
            kernel = ImageFilter.Kernel(size=(kernel_size, kernel_size),
                                        kernel=kernel.flatten().tolist(), scale=1)
            degraded = img.filter(kernel)
        elif deg_type == 3:  # Gaussian noise
            img_np = np.array(img).astype(np.float32) / 255.0
            noise = np.random.randn(*img_np.shape).astype(np.float32) * 0.15
            img_np = np.clip(img_np + noise, 0, 1)
            degraded = Image.fromarray((img_np * 255).astype(np.uint8))
        elif deg_type == 4:  # JPEG artifacts
            buffer = io.BytesIO()
            img.save(buffer, format="JPEG", quality=5)
            buffer.seek(0)
            degraded = Image.open(buffer).copy()

        if self.transform:
            degraded = self.transform(degraded)

        return degraded, deg_type

    @staticmethod
    def class_names():
        return ["Clean", "G.Blur", "M.Blur", "Noise", "JPEG"]


# Create dataset
raw_cifar = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)

# Preview degradations
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
sample_img = raw_cifar[42][0]  # PIL image

deg_names = DegradedImageDataset.DEGRADATION_NAMES
for i in range(5):
    # Create a temp dataset to get the degraded version
    temp_ds = DegradedImageDataset([(sample_img, 0)])
    degraded, label = temp_ds[i]

    axes[0, i].imshow(degraded)
    axes[0, i].set_title(deg_names[i], fontsize=10)
    axes[0, i].axis("off")

# Another example
sample_img2 = raw_cifar[100][0]
for i in range(5):
    temp_ds = DegradedImageDataset([(sample_img2, 0)])
    degraded, label = temp_ds[i]
    axes[1, i].imshow(degraded)
    axes[1, i].axis("off")

plt.suptitle("Synthetic Image Degradations", fontsize=14)
plt.tight_layout()
plt.show()

### Build Train/Val/Test Splits

We create the full degraded datasets from CIFAR-10's train and test sets, then split the training portion 90/10 for train/val. This gives us:
- **Training set** — the model learns from these
- **Validation set** — monitors overfitting during training (from same distribution as train)
- **Test set** — completely separate CIFAR-10 test images, used only once for final evaluation

In [ ]:
# Build train/val/test datasets
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_raw = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
test_raw  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True)

full_degraded = DegradedImageDataset(train_raw, transform=transform)
test_degraded = DegradedImageDataset(test_raw, transform=transform)

# Split
train_size = int(0.9 * len(full_degraded))
val_size = len(full_degraded) - train_size
train_data, val_data = random_split(full_degraded, [train_size, val_size],
                                     generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=256, shuffle=False)
test_loader  = DataLoader(test_degraded, batch_size=256, shuffle=False)

print(f"Train: {len(train_data)}")
print(f"Val:   {len(val_data)}")
print(f"Test:  {len(test_degraded)}")
print(f"Classes: {DegradedImageDataset.class_names()}")

---
## 2. Build the Model

We'll use a **ResNet-style CNN** — the same architecture pattern from Days 18 and 21. This is a proven choice for CIFAR-sized images because:
- **Residual blocks** (skip connections) allow deeper networks without degradation
- **BatchNorm** stabilizes training (Day 13)
- **AdaptiveAvgPool2d** makes the model resolution-independent
- **Dropout** before the final linear layer prevents overfitting (Day 12)

The `ResBlock` contains two 3×3 convolutions with a skip connection. Three stages progressively increase channels (64→128→256) while halving spatial dimensions (32→16→8).

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class DegradationClassifier(nn.Module):
    """Classifies image degradation type."""
    def __init__(self, num_classes=5):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(3, 64, 3, 1, 1, bias=False),
                                    nn.BatchNorm2d(64), nn.ReLU())
        self.layer1 = nn.Sequential(ResBlock(64, 64), ResBlock(64, 64))
        self.layer2 = nn.Sequential(ResBlock(64, 128, 2), ResBlock(128, 128))
        self.layer3 = nn.Sequential(ResBlock(128, 256, 2), ResBlock(256, 256))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(256, num_classes))

    def forward(self, x):
        x = self.conv1(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        return self.head(x)


model = DegradationClassifier(num_classes=5).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

---
## 3. Train

We use the standard training pipeline from Day 14:
- **CrossEntropyLoss** — standard for classification tasks
- **AdamW** — Adam with proper weight decay (Day 8)
- **CosineAnnealingLR** — smoothly decays the learning rate over 30 epochs (Day 13)
- **Best model checkpointing** — save whenever validation accuracy improves

The training and evaluation functions follow the same pattern you've used since Day 11.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        total_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.append(preds.cpu()); all_labels.append(labels.cpu())
    return total_loss / total, correct / total, torch.cat(all_preds), torch.cat(all_labels)


# Train
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0

print(f"{'Ep':>3} | {'TrLoss':>7} | {'TrAcc':>6} | {'VLoss':>7} | {'VAcc':>6}")
print("-" * 50)

for epoch in range(30):
    tl, ta = train_one_epoch(model, train_loader)
    vl, va, _, _ = evaluate(model, val_loader)
    scheduler.step()

    history["train_loss"].append(tl); history["val_loss"].append(vl)
    history["train_acc"].append(ta); history["val_acc"].append(va)

    if va > best_val_acc:
        best_val_acc = va
        torch.save(model.state_dict(), "best_degradation.pt")

    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f"{epoch+1:3d} | {tl:7.4f} | {ta:5.1%} | {vl:7.4f} | {va:5.1%}")

model.load_state_dict(torch.load("best_degradation.pt", map_location=device, weights_only=True))
print(f"\n✅ Best val accuracy: {best_val_acc:.2%}")

### Training Curves

Check the loss and accuracy curves for signs of healthy training — both should converge, and the train-val gap should be small (indicating good generalization):

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], label="Train"); ax1.plot(history["val_loss"], label="Val")
ax1.set_title("Loss"); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(history["train_acc"], label="Train"); ax2.plot(history["val_acc"], label="Val")
ax2.set_title("Accuracy"); ax2.legend(); ax2.grid(True, alpha=0.3)
for ax in [ax1, ax2]: ax.set_xlabel("Epoch")
plt.suptitle("Training History — Degradation Classifier", fontsize=14)
plt.tight_layout(); plt.show()

---
## 4. Evaluation & Analysis

Now we evaluate the best model on the test set. The **confusion matrix** is especially informative here — it reveals which degradation types the model confuses. For example, Gaussian blur and motion blur might be confused because both reduce sharpness.

From an optics perspective, this tells us which degradations are most visually distinct and which share similar spectral signatures.

In [ ]:
# Test set evaluation
test_loss, test_acc, all_preds, all_labels = evaluate(model, test_loader)
print(f"🎯 Test Accuracy: {test_acc:.2%}")

# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

class_names = DegradedImageDataset.class_names()
cm = confusion_matrix(all_labels.numpy(), all_preds.numpy())

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(5)); ax.set_xticklabels(class_names, rotation=45)
ax.set_yticks(range(5)); ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix")

# Add numbers
for i in range(5):
    for j in range(5):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, f"{cm[i,j]}", ha="center", va="center", color=color, fontsize=10)

plt.colorbar(im)
plt.tight_layout(); plt.show()

# Classification report
print("\n" + classification_report(all_labels.numpy(), all_preds.numpy(),
                                    target_names=class_names))

### Visualize Individual Predictions

Let's look at actual images with their true and predicted labels. Green titles mean the model got it right; red means wrong. This helps you develop intuition for what the model finds easy vs. challenging:

In [ ]:
# Visualize predictions on test images
fig, axes = plt.subplots(3, 5, figsize=(14, 9))

# Get some test images (unnormalized for display)
test_raw_imgs = torchvision.datasets.CIFAR10(root="./data", train=False, download=True)
test_degraded_vis = DegradedImageDataset(test_raw_imgs, transform=None)

np.random.seed(42)
sample_indices = np.random.choice(len(test_degraded_vis), 15, replace=False)

model.eval()
for i, idx in enumerate(sample_indices):
    row, col = divmod(i, 5)
    img_pil, true_label = test_degraded_vis[idx]

    # Get model prediction
    img_tensor = transform(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(img_tensor).argmax(1).item()

    axes[row, col].imshow(img_pil)
    color = "green" if pred == true_label else "red"
    axes[row, col].set_title(f"True: {class_names[true_label]}\nPred: {class_names[pred]}",
                             fontsize=9, color=color)
    axes[row, col].axis("off")

plt.suptitle("Degradation Classification Predictions", fontsize=14)
plt.tight_layout(); plt.show()

---
## 5. Feature Visualization — What Did the Model Learn?

The first convolutional layer's filters reveal what low-level features the model has learned to detect. Since our task is degradation classification (not object recognition), we expect these filters to be sensitive to **texture and frequency content** rather than shapes:

- **Smooth/uniform filters** → likely detecting blur (low-frequency content)
- **High-frequency/noisy-looking filters** → likely detecting noise patterns
- **Edge-like filters** → detecting blocking artifacts from JPEG compression

This connects directly to your optics background — these learned filters are essentially the model's version of spatial frequency analysis!

In [ ]:
# Extract first-layer filters
filters = model.conv1[0].weight.data.cpu()  # (64, 3, 3, 3)

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    f = filters[i]
    f = (f - f.min()) / (f.max() - f.min() + 1e-8)
    ax.imshow(f.permute(1, 2, 0))
    ax.axis("off")
    ax.set_title(f"F{i}", fontsize=7)
plt.suptitle("Learned First-Layer Filters (Degradation Classifier)", fontsize=14)
plt.tight_layout(); plt.show()

print("Notice: these filters likely learn to detect blur vs noise vs compression patterns!")

---
## 6. Reflection — Your 30-Day Journey 🗺️

### What you've built:
1. **Python & NumPy** — the tools of the trade
2. **PyTorch** — tensors, autograd, training loops
3. **MLPs** — your first neural networks
4. **CNNs** — convolutions, ResNets, feature visualization
5. **Transformers** — attention, ViT, patch embeddings
6. **Complete projects** — from data to trained model to evaluation

### Key principles that transcend specific architectures:
- **Gradient descent** is universal — it's how all models learn
- **Regularization** prevents overfitting — dropout, augmentation, weight decay
- **Residual connections** enable deep networks — used in both CNNs and Transformers
- **Normalization** stabilizes training — BatchNorm for CNNs, LayerNorm for Transformers
- **Attention** is a general mechanism — applicable to any sequence/spatial data

### Your optics advantage:
- Convolutions = spatial filters (you know this from PSF theory!)
- Feature hierarchies = multi-scale analysis (spatial frequencies!)
- Attention = adaptive, content-dependent filtering
- Image quality = your domain expertise applied to deep learning

---
## 📚 Where to Go Next

| Direction | Resources |
|---|---|
| **Object Detection** | YOLO, Faster R-CNN, DETR |
| **Segmentation** | U-Net, Mask R-CNN, SAM |
| **Generative Models** | GANs, VAEs, Diffusion models (Stable Diffusion) |
| **NLP / LLMs** | GPT architecture, fine-tuning LLMs, RAG |
| **Optical AI** | Lens design optimization, wavefront sensing, computational imaging |
| **Production ML** | ONNX export, TorchScript, model deployment |
| **Research** | Read papers on arXiv, reproduce results, attend conferences |

### Recommended next projects:
1. **Lens aberration classification** — use your own optical test images
2. **Defect detection** — surface inspection with CNNs
3. **Super-resolution** — enhance low-res images (optics + DL!)
4. **Depth estimation** — monocular depth from single images
5. **Fine-tune a foundation model** — adapt a pretrained ViT to your domain

---
## 🧪 Final Exercises

### Exercise 1: Add more degradations
Add chromatic aberration (shift R/G/B channels slightly) and vignetting (darken corners) as degradation types. Retrain.

### Exercise 2: Regression instead of classification
Instead of classifying degradation type, predict a continuous quality score (e.g., SSIM between clean and degraded). Use MSE loss.

### Exercise 3: Transfer learning approach
Use a pretrained ResNet-18 backbone and fine-tune it for degradation classification. Compare with your from-scratch model.

### Exercise 4: Real optical data
If you have access to real lens test images (e.g., from different f-numbers or focal lengths), build a classifier for optical quality.

In [ ]:
# Exercises: Your code here

---
## 📝 Final Key Takeaways

| Day | Milestone |
|---|---|
| Days 1–3 | Python, NumPy, ML fundamentals |
| Days 4–9 | PyTorch: tensors, autograd, training loop |
| Days 10–15 | Real datasets, MLPs, regularization, best practices |
| Days 16–21 | CNNs: convolutions → ResNet → from-scratch project |
| Day 22 | Self-attention — the paradigm shift |
| Days 23–26 | Transformer encoder, patch embeddings, ViT |
| Days 27–28 | CNN vs ViT comparison, hybrid architectures |
| Days 29–30 | Training toolkit & capstone project |

### The most important lesson:
**Deep learning is not magic — it's optimization.** Every model, from a simple linear regression to GPT-4, learns by:
1. Making a prediction (forward pass)
2. Computing how wrong it was (loss function)
3. Figuring out which parameters to blame (backpropagation)
4. Adjusting them slightly (gradient descent)

You now understand this pipeline deeply. Everything else is architecture design, data engineering, and engineering practice.

---

## 🎉 Congratulations!

You've completed the 30-day deep learning curriculum. You've gone from Python basics to building Vision Transformers from scratch. The journey from optics to deep learning is a powerful one — your intuition about spatial filtering, image formation, and signal processing gives you a unique perspective that most ML practitioners don't have.

**Keep building. Keep experimenting. Keep learning.** 🚀